# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
import re
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    ids = sorted(gutenberg.fileids())
except LookupError:
    nltk.download("gutenberg", raise_on_error=True)
    ids = sorted(gutenberg.fileids())

def normalizar_texto(texto):
    return re.findall(r"[^\W\d_]+", texto.lower(), flags=re.UNICODE)

def obtener_titulo(texto):
    encabezado = texto.splitlines()[0].strip().strip("[]")
    return re.split(r",?\s+by\s+", encabezado, maxsplit=1, flags=re.I)[0]

corpus = pd.DataFrame({"doc_id": ids, "Texto": [gutenberg.raw(d) for d in ids]})
corpus["Título"] = corpus["Texto"].map(obtener_titulo)
corpus["tokens"] = corpus["Texto"].map(normalizar_texto)
corpus = corpus.set_index("doc_id")[["Título", "Texto", "tokens"]]
display(corpus)  # La tabla conserva los textos y listas completos.

conteos = {d: Counter(tokens) for d, tokens in corpus["tokens"].items()}
df = Counter()
for conteo in conteos.values():
    df.update(conteo.keys())

N = len(corpus)
longitudes = corpus["tokens"].map(len)
media = longitudes.mean()
estadisticas = pd.DataFrame.from_dict(df, orient="index", columns=["DF"]).sort_index()
estadisticas.index.name = "Término"
estadisticas["IDF TF-IDF"] = np.log(N / estadisticas["DF"]) + 1
estadisticas["IDF BM25"] = np.log1p((N - estadisticas["DF"] + 0.5) / (estadisticas["DF"] + 0.5))
print(f"{N} documentos | {len(estadisticas)} términos | Longitud media: {media:,.2f} tokens")
display(estadisticas.loc[["the", "sea", "whale", "harpoon"]])

,Título,Texto,tokens
doc_id,,,
austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAP...,"[emma, by, jane, austen, volume, i, chapter, i..."
austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter ...,"[persuasion, by, jane, austen, chapter, sir, w..."
austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\...,"[sense, and, sensibility, by, jane, austen, ch..."
bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of...,"[the, king, james, bible, the, old, testament,..."
blake-poems.txt,Poems,[Poems by William Blake 1789]\n\n \nSONGS OF I...,"[poems, by, william, blake, songs, of, innocen..."
bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Brya...,"[stories, to, tell, to, children, by, sara, co..."
burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. ...,"[the, adventures, of, buster, bear, by, thornt..."
carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Car...,"[alice, s, adventures, in, wonderland, by, lew..."
chesterton-ball.txt,The Ball and The Cross,[The Ball and The Cross by G.K. Chesterton 190...,"[the, ball, and, the, cross, by, g, k, chester..."


18 documentos | 41509 términos | Longitud media: 118,670.78 tokens


,DF,IDF TF-IDF,IDF BM25
Término,,,
the,18,1.000000,0.026668
sea,16,1.117783,0.141079
whale,6,2.098612,1.072637
harpoon,2,3.197225,2.028148


**Cómo leer los dos IDF.** El DF de un término cuenta los documentos que lo contienen. TF-IDF usa $\ln(N/df)+1$, como en la práctica 03. BM25 usa $\ln(1+(N-df+0.5)/(df+0.5))$. Para un término presente en todos los libros, el primer IDF vale 1 y el segundo queda cerca de cero. La tabla anterior permite contrastar el artículo *the* con el sustantivo *harpoon*.

**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
consultas = {
    "the": "Medir qué libros recupera una búsqueda por un artículo presente en toda la colección.",
    "harpoon": "Localizar libros que mencionen el arpón.",
    "whale harpoon sea": "Buscar narraciones sobre la caza de ballenas en el mar.",
    "rabbit queen wonderland": "Localizar la historia del conejo y la reina en el País de las Maravillas.",
    "prophet covenant israel": "Encontrar pasajes sobre profetas y la alianza de Israel.",
    "marriage family fortune": "Encontrar relatos que relacionen matrimonio, familia y fortuna.",
    "king ghost hamlet": "Buscar la tragedia que relaciona a Hamlet con el espectro de un rey.",
    "heaven and earth": "Encontrar referencias al cielo y la tierra."
}
filas = []
for q, intencion in consultas.items():
    tokens = normalizar_texto(q)
    cobertura = ", ".join(f"{t} ({df[t]}/{N})" for t in tokens)
    filas.append([q, intencion, len(tokens), cobertura])
display(pd.DataFrame(filas, columns=["Consulta", "Necesidad de información", "Tokens", "DF por término"]))

,Consulta,Necesidad de información,Tokens,DF por término
0,the,Medir qué libros recupera una búsqueda por un ...,1,the (18/18)
1,harpoon,Localizar libros que mencionen el arpón.,1,harpoon (2/18)
2,whale harpoon sea,Buscar narraciones sobre la caza de ballenas e...,3,"whale (6/18), harpoon (2/18), sea (16/18)"
3,rabbit queen wonderland,Localizar la historia del conejo y la reina en...,3,"rabbit (6/18), queen (15/18), wonderland (2/18)"
4,prophet covenant israel,Encontrar pasajes sobre profetas y la alianza ...,3,"prophet (6/18), covenant (4/18), israel (5/18)"
5,marriage family fortune,"Encontrar relatos que relacionen matrimonio, f...",3,"marriage (12/18), family (13/18), fortune (12/18)"
6,king ghost hamlet,Buscar la tragedia que relaciona a Hamlet con ...,3,"king (15/18), ghost (9/18), hamlet (6/18)"
7,heaven and earth,Encontrar referencias al cielo y la tierra.,3,"heaven (13/18), and (18/18), earth (17/18)"


Se incluyen consultas de una palabra para comparar dos extremos de cobertura. En la consulta náutica, *sea* aparece en 16 libros y *harpoon* en dos. Esto permite observar qué añade un término raro a una consulta amplia. Las consultas en inglés pasan por la misma tokenización que los libros. Las coincidencias se calculan por términos individuales.

In [3]:
def usar_tokens(tokens):
    return tokens

vectorizador = TfidfVectorizer(analyzer=usar_tokens, smooth_idf=False,
                               sublinear_tf=False, use_idf=True, norm=None)
matriz_tfidf = vectorizador.fit_transform(corpus["tokens"])
assert issparse(matriz_tfidf)
np.testing.assert_allclose(vectorizador.idf_,
    estadisticas.loc[vectorizador.get_feature_names_out(), "IDF TF-IDF"])

def puntuar_coseno(tokens):
    consulta = vectorizador.transform([tokens])
    # El único resultado denso es un vector con una puntuación por libro.
    return cosine_similarity(consulta, matriz_tfidf).ravel()

def aporte_bm25(termino, doc_id, k1, b):
    f = conteos[doc_id][termino]
    if f == 0:
        return 0.0
    ajuste = 1 - b + b * longitudes[doc_id] / media
    factor_tf = f * (k1 + 1) / (f + k1 * ajuste)
    return estadisticas.at[termino, "IDF BM25"] * factor_tf

def puntuar_bm25(tokens, k1=1.2, b=0.75):
    terminos = list(dict.fromkeys(tokens))  # Un aporte por término distinto.
    return [sum(aporte_bm25(t, d, k1, b) for t in terminos) for d in ids]

def ranking(puntuaciones):
    positivos = [(d, float(p)) for d, p in zip(ids, puntuaciones) if p > 0]
    return sorted(positivos, key=lambda par: (-par[1], par[0]))[:10]

def buscar(consulta):
    tokens = normalizar_texto(consulta)
    return ranking(puntuar_coseno(tokens)), ranking(puntuar_bm25(tokens))

resultados = {q: buscar(q) for q in consultas}
consulta_ausente = "zzqxnonexistent"
assert not any(t in df for t in normalizar_texto(consulta_ausente))
sin_tf, sin_bm = buscar(consulta_ausente)
assert sin_tf == sin_bm == []
print("Control fuera del vocabulario:", consulta_ausente, "| TF-IDF:", sin_tf, "| BM25:", sin_bm)

Control fuera del vocabulario: zzqxnonexistent | TF-IDF: [] | BM25: []


**Cálculo del score.** TF-IDF compara los vectores completos mediante el coseno. BM25 suma, para cada término distinto de la consulta, el aporte

$$
\mathrm{IDF}_{BM25}(t)\,\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1(1-b+b\,|d|/\overline L)}.
$$

Aquí $|d|$ es el número total de tokens y $\overline L$ la longitud media de la colección. La función devuelve cero cuando el término está ausente. Al crecer mucho $f$, el factor de frecuencia se aproxima a $k_1+1$. La matriz dispersa, los DF, los IDF y las longitudes se calculan una sola vez y se reutilizan en todas las búsquedas.

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [4]:
def tabla_metodo(lista, metodo):
    etiqueta = "ID y título " + metodo
    medida = "Coseno" if metodo == "TF-IDF" else "Score BM25"
    return pd.DataFrame([
        {"Posición": i, etiqueta: f"{d} · {corpus.at[d, 'Título']}",
         medida: score}
        for i, (d, score) in enumerate(lista, 1)
    ], columns=["Posición", etiqueta, medida]).set_index("Posición")

resumen = []
for q, (r_tf, r_bm) in resultados.items():
    print(q)
    comparacion = tabla_metodo(r_tf, "TF-IDF").join(tabla_metodo(r_bm, "BM25"), how="outer")
    with pd.option_context("display.max_colwidth", None):
        display(comparacion)
    ids_tf, ids_bm = {d for d, _ in r_tf}, {d for d, _ in r_bm}
    resumen.append([q, len(r_tf), len(r_bm), len(ids_tf & ids_bm),
                    next((d for d, _ in r_tf), None), next((d for d, _ in r_bm), None)])
display(pd.DataFrame(resumen, columns=["Consulta", "N TF-IDF", "N BM25", "IDs comunes",
                                       "Primero TF-IDF", "Primero BM25"]).set_index("Consulta"))

the


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,whitman-leaves.txt · Leaves of Grass,0.723446,bible-kjv.txt · The King James Bible,0.058664
2,melville-moby_dick.txt · Moby Dick,0.669970,whitman-leaves.txt · Leaves of Grass,0.058663
3,bryant-stories.txt · Stories to Tell to Children,0.666026,melville-moby_dick.txt · Moby Dick,0.058662
4,chesterton-brown.txt · The Wisdom of Father Brown,0.633625,chesterton-brown.txt · The Wisdom of Father Brown,0.058659
5,bible-kjv.txt · The King James Bible,0.619686,chesterton-ball.txt · The Ball and The Cross,0.058659
6,blake-poems.txt · Poems,0.611702,bryant-stories.txt · Stories to Tell to Children,0.058659
7,chesterton-thursday.txt · The Man Who Was Thursday,0.588002,chesterton-thursday.txt · The Man Who Was Thursday,0.058658
8,chesterton-ball.txt · The Ball and The Cross,0.582681,edgeworth-parents.txt · The Parent's Assistant,0.058658
9,carroll-alice.txt · Alice's Adventures in Wonderland,0.547424,austen-persuasion.txt · Persuasion,0.058654


harpoon


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,melville-moby_dick.txt · Moby Dick,0.011281,melville-moby_dick.txt · Moby Dick,4.349960
2,whitman-leaves.txt · Leaves of Grass,0.000229,whitman-leaves.txt · Leaves of Grass,1.976274


whale harpoon sea


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,melville-moby_dick.txt · Moby Dick,0.078590,melville-moby_dick.txt · Moby Dick,7.015046
2,whitman-leaves.txt · Leaves of Grass,0.004886,whitman-leaves.txt · Leaves of Grass,4.170131
3,bryant-stories.txt · Stories to Tell to Children,0.004622,bryant-stories.txt · Stories to Tell to Children,2.567654
4,milton-paradise.txt · Paradise Lost,0.002690,shakespeare-hamlet.txt · The Tragedie of Hamlet,2.150998
5,chesterton-brown.txt · The Wisdom of Father Brown,0.002638,chesterton-ball.txt · The Ball and The Cross,1.917374
6,chesterton-ball.txt · The Ball and The Cross,0.002350,bible-kjv.txt · The King James Bible,1.066346
7,chesterton-thursday.txt · The Man Who Was Thursday,0.001826,chesterton-brown.txt · The Wisdom of Father Brown,0.306147
8,shakespeare-hamlet.txt · The Tragedie of Hamlet,0.001483,milton-paradise.txt · Paradise Lost,0.305576
9,carroll-alice.txt · Alice's Adventures in Wonderland,0.001464,chesterton-thursday.txt · The Man Who Was Thursday,0.304077


... (7 outputs left out)

,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,shakespeare-hamlet.txt · The Tragedie of Hamlet,0.091214,shakespeare-hamlet.txt · The Tragedie of Hamlet,4.281331
2,shakespeare-macbeth.txt · The Tragedie of Macbeth,0.017176,chesterton-ball.txt · The Ball and The Cross,3.063530
3,bryant-stories.txt · Stories to Tell to Children,0.012916,melville-moby_dick.txt · Moby Dick,2.558355
4,bible-kjv.txt · The King James Bible,0.012713,whitman-leaves.txt · Leaves of Grass,2.460428
5,carroll-alice.txt · Alice's Adventures in Wonderland,0.009972,edgeworth-parents.txt · The Parent's Assistant,1.945455
6,shakespeare-caesar.txt · The Tragedie of Julius Caesar,0.003683,bible-kjv.txt · The King James Bible,1.888345
7,milton-paradise.txt · Paradise Lost,0.003370,shakespeare-caesar.txt · The Tragedie of Julius Caesar,1.818750
8,melville-moby_dick.txt · Moby Dick,0.002068,shakespeare-macbeth.txt · The Tragedie of Macbeth,1.774801
9,chesterton-brown.txt · The Wisdom of Father Brown,0.001881,chesterton-brown.txt · The Wisdom of Father Brown,1.686545


heaven and earth


,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
Posición,,,,
1,milton-paradise.txt · Paradise Lost,0.328599,milton-paradise.txt · Paradise Lost,0.989041
2,blake-poems.txt · Poems,0.266700,bible-kjv.txt · The King James Bible,0.982242
3,bible-kjv.txt · The King James Bible,0.264671,chesterton-ball.txt · The Ball and The Cross,0.970206
4,bryant-stories.txt · Stories to Tell to Children,0.209550,whitman-leaves.txt · Leaves of Grass,0.962905
5,whitman-leaves.txt · Leaves of Grass,0.205955,blake-poems.txt · Poems,0.959615
6,austen-persuasion.txt · Persuasion,0.177321,melville-moby_dick.txt · Moby Dick,0.956271
7,shakespeare-macbeth.txt · The Tragedie of Macbeth,0.166786,chesterton-thursday.txt · The Man Who Was Thursday,0.948550
8,chesterton-ball.txt · The Ball and The Cross,0.165792,chesterton-brown.txt · The Wisdom of Father Brown,0.886393
9,edgeworth-parents.txt · The Parent's Assistant,0.163232,edgeworth-parents.txt · The Parent's Assistant,0.857907


,N TF-IDF,N BM25,IDs comunes,Primero TF-IDF,Primero BM25
Consulta,,,,,
the,10,10,8,whitman-leaves.txt,bible-kjv.txt
harpoon,2,2,2,melville-moby_dick.txt,melville-moby_dick.txt
whale harpoon sea,10,10,10,melville-moby_dick.txt,melville-moby_dick.txt
rabbit queen wonderland,10,10,10,carroll-alice.txt,carroll-alice.txt
prophet covenant israel,9,9,9,bible-kjv.txt,bible-kjv.txt
marriage family fortune,10,10,8,austen-persuasion.txt,austen-sense.txt
king ghost hamlet,10,10,7,shakespeare-hamlet.txt,shakespeare-hamlet.txt
heaven and earth,10,10,7,milton-paradise.txt,milton-paradise.txt


In [5]:
consulta_estudio = "marriage family fortune"
terminos = normalizar_texto(consulta_estudio)
libros = ["austen-persuasion.txt", "austen-sense.txt", "austen-emma.txt"]
r_tf, r_bm = resultados[consulta_estudio]

def posiciones(lista):
    return {d: i for i, (d, _) in enumerate(lista, start=1)}

display(pd.DataFrame([
    [d, longitudes[d], posiciones(r_tf)[d], dict(r_tf)[d], posiciones(r_bm)[d], dict(r_bm)[d]]
    for d in libros
], columns=["ID", "Longitud", "Puesto TF-IDF", "Coseno", "Puesto BM25", "Score BM25"]))

def detalle_aportes(documentos, k1, b):
    filas = []
    for d in documentos:
        for t in terminos:
            filas.append([d, longitudes[d], t, conteos[d][t], df[t],
                          estadisticas.at[t, "IDF BM25"], aporte_bm25(t, d, k1, b)])
    return pd.DataFrame(filas, columns=["ID", "Longitud", "Término", "TF", "DF", "IDF BM25", "Aporte"])

detalle = detalle_aportes(libros, 1.2, 0.75)
display(detalle)
for d, suma in detalle.groupby("ID")["Aporte"].sum().items():
    np.testing.assert_allclose(suma, dict(r_bm)[d])

,ID,Longitud,Puesto TF-IDF,Coseno,Puesto BM25,Score BM25
0,austen-persuasion.txt,84123,1,0.013038,2,2.525673
1,austen-sense.txt,120734,2,0.011173,1,2.530000
2,austen-emma.txt,161975,3,0.007676,3,2.506258


,ID,Longitud,Término,TF,DF,IDF BM25,Aporte
0,austen-persuasion.txt,84123,marriage,27,12,0.418710,0.890236
1,austen-persuasion.txt,84123,family,80,13,0.341749,0.743135
2,austen-persuasion.txt,84123,fortune,29,12,0.418710,0.892302
3,austen-sense.txt,120734,marriage,44,12,0.418710,0.896397
4,austen-sense.txt,120734,family,83,13,0.341749,0.740996
5,austen-sense.txt,120734,fortune,38,12,0.418710,0.892608
6,austen-emma.txt,161975,marriage,35,12,0.418710,0.882620
7,austen-emma.txt,161975,family,77,13,0.341749,0.737215
8,austen-emma.txt,161975,fortune,39,12,0.418710,0.886424


**Por qué se invierten los dos primeros.** Para el coseno, el vector de *Persuasion* está más alineado con la consulta (0.013038) que el de *Sense and Sensibility* (0.011173). BM25 da a *Sense and Sensibility* una pequeña ventaja en *marriage* (44 apariciones frente a 27) y en *fortune* (38 frente a 29). *Persuasion*, más breve, obtiene un aporte ligeramente mayor por *family* pese a contener 80 apariciones frente a 83. Al sumar los tres aportes, *Sense and Sensibility* queda primero por aproximadamente 0.00433. Las diferencias son pequeñas porque los tres términos ya se repiten decenas de veces en ambos libros.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [6]:
configuraciones = [("Base", 1.2, 0.75), ("Sin ajuste de longitud", 1.2, 0.0),
                   ("Mayor efecto de repeticiones", 2.0, 0.75)]
experimentos = []
for nombre, k1, b in configuraciones:
    puntuaciones = puntuar_bm25(terminos, k1, b)
    orden = posiciones(ranking(puntuaciones))
    detalle = detalle_aportes(libros, k1, b)
    tabla = detalle.pivot(index="ID", columns="Término", values="Aporte").reindex(libros)
    tabla = tabla[terminos].rename_axis(columns=None)
    tabla["Score BM25"] = tabla.sum(axis=1)
    tabla["Puesto"] = [orden[d] for d in libros]
    tabla["Longitud"] = longitudes
    tabla.insert(0, "Configuración", nombre)
    tabla.insert(1, "k1", k1)
    tabla.insert(2, "b", b)
    np.testing.assert_allclose(tabla["Score BM25"], pd.Series(puntuaciones, index=ids).loc[libros])
    experimentos.append(tabla.reset_index())
display(pd.concat(experimentos, ignore_index=True).set_index(["Configuración", "ID"]).round(6))

k1     b  ...  Puesto  Longitud
Configuración                ID                                ...                  
Base                         austen-persuasion.txt  1.2  0.75  ...       2     84123
                             austen-sense.txt       1.2  0.75  ...       1    120734
                             austen-emma.txt        1.2  0.75  ...       3    161975
Sin ajuste de longitud       austen-persuasion.txt  1.2  0.00  ...       3     84123
                             austen-sense.txt       1.2  0.00  ...       1    120734
                             austen-emma.txt        1.2  0.00  ...       2    161975
Mayor efecto de repeticiones austen-persuasion.txt  2.0  0.75  ...       2     84123
                             austen-sense.txt       2.0  0.75  ...       1    120734
                             austen-emma.txt        2.0  0.75  ...       3    161975

[9 rows x 8 columns]

**Efecto de la longitud.** *Persuasion* tiene 84 123 tokens, por debajo de la media de 118 670.78. *Emma* tiene 161 975, por encima. Con `b=0` desaparece el ajuste por longitud. *Emma* sube del tercer puesto al segundo y *Persuasion* baja del segundo al tercero. En *Emma*, el aporte de *marriage* sube de 0.88262 a 0.89063. En *Persuasion* baja de 0.89024 a 0.88196.

**Efecto de las repeticiones.** Al pasar de `k1=1.2` a `k1=2.0` con `b=0.75`, sube el límite del factor de frecuencia de 2.2 a 3.0 y la saturación es más gradual. Para *marriage*, la ventaja de *Sense and Sensibility* sobre *Persuasion* pasa de aproximadamente 0.00616 a 0.01346. Ambos libros conservan sus posiciones porque el cambio amplía la ventaja del primero en la suma de los tres términos.